In [ ]:
import pandas as pd

# 파일 경로 및 시트명
file_path = r"C:\Users\manid\Desktop\data_study\mutdae_2025.xlsx"
sheet_name = "campus_data"

# 엑셀 데이터 불러오기
df = pd.read_excel(file_path, sheet_name=sheet_name)

# 데이터 기본 정보 출력
print("📌 데이터프레임 정보:")
print(df.info())

print("\n📌 데이터 미리보기 (상위 5개 행):")
print(df.head())

print("\n📌 컬럼별 요약 통계:")
print(df.describe(include="all"))


In [ ]:
import pandas as pd

# 파일 경로 및 시트명
file_path = r"C:\Users\manid\Desktop\data_study\mutdae_2025.xlsx"
sheet_name = "campus_data"

# 엑셀 데이터 불러오기
df = pd.read_excel(file_path, sheet_name=sheet_name)

# 컬럼별 결측값 개수 확인
missing_values = df.isnull().sum()
print("📌 컬럼별 결측값 개수:")
print(missing_values)

# 결측값이 있는 컬럼만 출력
missing_cols = missing_values[missing_values > 0]
if not missing_cols.empty:
    print("\n📌 결측값이 있는 컬럼 목록:")
    print(missing_cols)

# 결측값을 'null'로 대체
df.fillna('null', inplace=True)

# 변환 후 결측값 확인
print("\n✅ 결측값을 'null'로 대체한 후 확인:")
print(df.isnull().sum())

# 변경된 데이터 저장 (필요할 경우)
output_path = r"C:\Users\manid\Desktop\data_study\mutdae_2025_filled.xlsx"
df.to_excel(output_path, index=False)
print(f"\n💾 변환된 데이터가 저장되었습니다: {output_path}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 파일 경로
file_path = r"C:\Users\manid\Desktop\data_study\mutdae_2025_filled.xlsx"

# 엑셀 데이터 불러오기
df = pd.read_excel(file_path)

# '멋사 기수' → 'LikeLion Cohort'로 변경
df.rename(columns={'멋사 기수': 'LikeLion Cohort', '학교명': 'School Name'}, inplace=True)

# Cohort별 회원 수 계산
member_counts = df['LikeLion Cohort'].value_counts()

# Cohort별 학교 수 계산 (중복 제거)
school_counts = df.groupby('LikeLion Cohort')['School Name'].nunique()

# Cohort를 5기부터 12기까지 정렬
cohort_order = [f"{i}기" for i in range(5, 13)]
member_counts = member_counts.reindex(cohort_order).fillna(0)
school_counts = school_counts.reindex(cohort_order).fillna(0)

# 막대 그래프 시각화 함수 (정렬된 X축 적용)
def plot_bar_chart(data, title, xlabel, ylabel, color='skyblue'):
    plt.figure(figsize=(10, 5))
    bars = plt.bar(data.index, data.values, color=color)
    
    # 막대 위에 레이블 추가
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{int(bar.get_height())}', 
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

# 1. LikeLion Cohort별 회원 수 그래프
plot_bar_chart(member_counts, "Number of Members per LikeLion Cohort", "LikeLion Cohort", "Number of Members", color='cornflowerblue')

# 2. LikeLion Cohort별 학교 수 그래프
plot_bar_chart(school_counts, "Number of Schools per LikeLion Cohort", "LikeLion Cohort", "Number of Schools", color='lightcoral')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 파일 경로
file_path = r"C:\Users\manid\Desktop\data_study\mutdae_2025_filled.xlsx"

# 엑셀 데이터 불러오기
df = pd.read_excel(file_path)

# '멋사 기수' → 'LikeLion Cohort', '학교명' → 'School Name' 변경
df.rename(columns={'멋사 기수': 'LikeLion Cohort', '학교명': 'School Name'}, inplace=True)

# 각 기수별 학교별 학생 수 계산
cohort_school_counts = df.groupby(['LikeLion Cohort', 'School Name']).size().reset_index(name='Students per School')

# 기수 순서 지정 (5기~12기)
cohort_order = [f"{i}기" for i in range(5, 13)]
cohort_school_counts['LikeLion Cohort'] = pd.Categorical(cohort_school_counts['LikeLion Cohort'], categories=cohort_order, ordered=True)

# 박스플롯 시각화
plt.figure(figsize=(12, 6))
ax = sns.boxplot(data=cohort_school_counts, x='LikeLion Cohort', y='Students per School', palette="Set3")

# 중앙값, 최소, 최대값 계산 및 레이블 추가
grouped = cohort_school_counts.groupby('LikeLion Cohort')['Students per School']

for i, cohort in enumerate(cohort_order):
    if cohort in grouped.groups:  # 기수별 데이터가 존재하면 실행
        values = grouped.get_group(cohort)

        # 중앙값, 최소값, 최대값 계산
        median_val = values.median() if not values.empty else 0
        min_val = values.min() if not values.empty else 0
        max_val = values.max() if not values.empty else 0

        # 레이블 추가
        ax.text(i, median_val, f'{median_val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='blue')
        ax.text(i, min_val, f'{min_val:.1f}', ha='center', va='top', fontsize=10, fontweight='bold', color='red')
        ax.text(i, max_val, f'{max_val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='green')
    else:
        # 데이터가 없는 경우 0으로 표시
        ax.text(i, 0, '0.0', ha='center', va='bottom', fontsize=10, fontweight='bold', color='gray')

# 그래프 설정
plt.xlabel("LikeLion Cohort")
plt.ylabel("Students per School")
plt.title("Distribution of Students per School in Each LikeLion Cohort")
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 그래프 출력
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 파일 경로 (Jupyter 환경에 맞게 수정)
file_path = r"C:\Users\manid\Desktop\data_study\mutdae_2025_filled.xlsx"

# 엑셀 데이터 불러오기
df = pd.read_excel(file_path)

# '멋사 기수' → 'LikeLion Cohort', '학교명' → 'School Name' 변경
df.rename(columns={'멋사 기수': 'LikeLion Cohort', '학교명': 'School Name'}, inplace=True)

# 기수별 총 학생 수 계산
student_counts = df['LikeLion Cohort'].value_counts().sort_index()

# 기수별 학교 수 계산 (중복 제거)
school_counts = df.groupby('LikeLion Cohort')['School Name'].nunique().sort_index()

# 기수별 평균 학교당 학생 수 계산
avg_students_per_school = (student_counts / school_counts).dropna()

# Jupyter에서 한글 폰트 설정 (Windows 환경)
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')

# 막대 그래프 시각화
plt.figure(figsize=(10, 5))
bars = plt.bar(avg_students_per_school.index, avg_students_per_school.values, color='mediumseagreen')

# 막대 위에 레이블 추가
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.2f}', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# 그래프 설정
plt.xlabel("LikeLion Cohort")
plt.ylabel("Avg. Students per School")
plt.title("Average Number of Students per School in Each LikeLion Cohort")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 그래프 출력
plt.show()
